# Pipeline de nós: MACCROBAT2020 (brat) -> split -> PubMedBERT -> classificador de nós

Fluxo deste notebook:

| Etapa | Módulo |
|---|---|
| 1. Leitura do brat (.txt/.ann) | `brat_reader.py` |
| 2. Split treino/val/teste por caso | `split_cases.py` |
| 3. Embeddings PubMedBERT por subtoken | `embeddings_bert.py` |
| 4. Monta X/y de nós (cache em .npz) | `build_datasets.py` |
| 5. Classificador de nós (LogReg) + avaliação por entidade | `node_classifier.py` |

Para comparar com outro classificador, troque só a seção 5 — o `X`/`y` continuam no mesmo formato.

In [1]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()                       # este notebook fica em notebooks/
sys.path.insert(0, str(ROOT / "models" / "pre_processamento"))
sys.path.insert(0, str(ROOT / "models" / "P2_Adaptado"))

import brat_reader, split_cases, build_datasets, evaluation
from embeddings_bert import BertEmbedder
from node_classifier import NodeClassifier, fix_bio, bio_to_entities

DATA_DIR = ROOT / "data" / "treinamento" / "MACCROBAT2020"
OUT_DIR = ROOT / "data" / "treinamento" / "embeddings"

c:\Users\hidal\Documents\GitHub\Diplotokens-P1\project 1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Leitura do brat e divisão treino/val/teste

In [8]:
docs = brat_reader.load_docs(DATA_DIR)
print(len(docs), "casos | entidades com posição desalinhada:", brat_reader.check_offsets(docs))

splits = split_cases.get_splits([d.id for d in docs], OUT_DIR / "splits.json", seed=42)
print({k: len(v) for k, v in splits.items()})

200 casos | entidades com posição desalinhada: 0
{'train': 140, 'val': 30, 'test': 30}


## 2. Embeddings PubMedBERT (roda uma vez; depois vem do cache em disco)

A primeira execução baixa o modelo do HuggingFace e processa todos os casos;
chamadas seguintes reaproveitam o `.npz` salvo em `OUT_DIR`.

In [11]:
embedder = BertEmbedder(mask_digits=False)
build_datasets.build_token_dataset(docs, splits, embedder, OUT_DIR)

train = build_datasets.load_token_dataset(OUT_DIR, "train", embedder.name)
val = build_datasets.load_token_dataset(OUT_DIR, "val", embedder.name)
test = build_datasets.load_token_dataset(OUT_DIR, "test", embedder.name)
print("train:", train["X"].shape, "| val:", val["X"].shape, "| test:", test["X"].shape)

I1119 15:46:31.556259 4784 _client.py:1084] HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
I1119 15:46:31.703081 4784 _client.py:1084] HTTP Request: GET https://huggingface.co/api/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext "HTTP/1.1 200 OK"
I1119 15:46:32.052069 4784 _client.py:1084] HTTP Request: GET https://huggingface.co/api/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
I1119 15:46:32.198921 4784 _client.py:1084] HTTP Request: GET https://huggingface.co/api/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
I1119 15:46:32.411683 4784 _client.py:1084] HTTP Request: GET https://huggingface.co/api/models/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext "HTTP/1.1 200 OK"
I1119 15:46:37.282818 4784 _client.py:1084] HTTP Request: HEAD 

[train] cache encontrado: train_BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.npz
[val] cache encontrado: val_BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.npz
[test] cache encontrado: test_BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext.npz
train: (82488, 768) | val: (15947, 768) | test: (16362, 768)


## 3. Classificador de nós

`NodeClassifier` é um `LogisticRegression` que recebe o `X` denso do BERT diretamente
(sem `DictVectorizer`, diferente do P2). A avaliação é por ENTIDADE, não por subtoken.

In [2]:
embedder = BertEmbedder(mask_digits=False)   # não baixa/roda nada ainda, só monta o objeto
train = build_datasets.load_token_dataset(OUT_DIR, "train", embedder.name)
val   = build_datasets.load_token_dataset(OUT_DIR, "val",   embedder.name)
test  = build_datasets.load_token_dataset(OUT_DIR, "test",  embedder.name)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35313.36it/s]
[transformers] BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
clf = NodeClassifier(C=1.0, max_iter=1000).fit(train["X"], train["y"])

pred_val = clf.predict(val["X"])
rep_val = clf.evaluate(val["y"], pred_val, val["doc"])
clf.print_report(rep_val)

tipo                              P      R     F1      n
Activity                      0.034  0.200  0.059      5
Administration                0.655  0.792  0.717     24
Age                           0.966  0.933  0.949     30
Area                          0.167  0.667  0.267      3
Biological_attribute          0.000  0.000  0.000      5
Biological_structure          0.355  0.532  0.426    419
Clinical_event                0.567  0.800  0.664    100
Color                         1.000  1.000  1.000      3
Coreference                   0.049  0.128  0.071     47
Date                          0.220  0.444  0.294     90
Detailed_description          0.201  0.387  0.265    401
Diagnostic_procedure          0.537  0.653  0.590    681
Disease_disorder              0.195  0.377  0.257    167
Distance                      0.282  0.500  0.361     22
Dosage                        0.510  0.735  0.602     34
Duration                      0.190  0.333  0.242     36
Family_history                0

## 4. Pós-processamento: BIO -> entidades

`fix_bio` corrige sequências inválidas (um `I-X` sem `B-X` antes vira `B-X`).
`bio_to_entities` converte as tags em entidades com posição de caractere — é o formato
que o classificador de arestas espera como entrada.

In [6]:
pred_val_fixed = fix_bio(list(pred_val))

# entidades do primeiro documento da validação (como exemplo)
doc0 = val["doc"][0]
mask0 = val["doc"] == doc0
offsets0 = val["offsets"][mask0]
tags0 = pred_val_fixed[:mask0.sum()]  # ajuste: ver nota abaixo

ents0 = bio_to_entities(val["offsets"][mask0], [t for t, m in zip(pred_val_fixed, mask0) if m])
print(f"doc {doc0}: {len(ents0)} entidades previstas")
ents0[:10]

doc 24781756: 250 entidades previstas


[{'label': 'Family_history',
  'char_start': 0,
  'char_end': 3,
  'token_start': 0,
  'token_end': 1},
 {'label': 'Family_history',
  'char_start': 4,
  'char_end': 12,
  'token_start': 1,
  'token_end': 2},
 {'label': 'History',
  'char_start': 13,
  'char_end': 26,
  'token_start': 2,
  'token_end': 5},
 {'label': 'Sex',
  'char_start': 27,
  'char_end': 31,
  'token_start': 5,
  'token_end': 6},
 {'label': 'History',
  'char_start': 31,
  'char_end': 32,
  'token_start': 6,
  'token_end': 7},
 {'label': 'History',
  'char_start': 37,
  'char_end': 40,
  'token_start': 8,
  'token_end': 9},
 {'label': 'Family_history',
  'char_start': 41,
  'char_end': 53,
  'token_start': 9,
  'token_end': 11},
 {'label': 'History',
  'char_start': 54,
  'char_end': 61,
  'token_start': 11,
  'token_end': 13},
 {'label': 'Family_history',
  'char_start': 62,
  'char_end': 64,
  'token_start': 13,
  'token_end': 14},
 {'label': 'Personal_background',
  'char_start': 65,
  'char_end': 76,
  'token_st

**Nota:** o recorte acima assume que `val["doc"]` preserva a ordem original dos subtokens
de cada documento (sem embaralhar). Se precisar iterar por TODOS os documentos, use
`build_datasets.doc_slices` (já existe no módulo) para obter os índices de cada caso.

In [7]:
# Avaliação final no conjunto de teste (só rodar depois de fechar o modelo na validação)
pred_test = clf.predict(test["X"])
rep_test = clf.evaluate(test["y"], pred_test, test["doc"])
clf.print_report(rep_test)

tipo                              P      R     F1      n
Activity                      0.149  0.333  0.206     21
Administration                0.579  0.647  0.611     17
Age                           0.968  0.968  0.968     31
Area                          0.083  0.500  0.143      2
Biological_structure          0.360  0.533  0.430    433
Clinical_event                0.569  0.780  0.658    100
Color                         0.667  1.000  0.800      6
Coreference                   0.216  0.433  0.289     67
Date                          0.316  0.596  0.413     94
Detailed_description          0.147  0.343  0.206    417
Diagnostic_procedure          0.486  0.645  0.554    625
Disease_disorder              0.243  0.570  0.341    165
Distance                      0.129  0.308  0.182     13
Dosage                        0.411  0.697  0.517     33
Duration                      0.214  0.500  0.300     36
Family_history                0.000  0.000  0.000      7
Frequency                     0

## Observações

- O `X` de nós aqui são vetores PubMedBERT (768-d) por subtoken; o `y` são as tags BIO
  vindas diretamente do gold do MACCROBAT (via `brat_reader` + `build_datasets`).
- Ainda não há mapeamento MACCROBAT -> schema do P2 (Disease, Symptom, Exam...) neste
  notebook — os tipos aqui são os originais do MACCROBAT (Sign_symptom, Disease_disorder,
  Diagnostic_procedure, etc.). Esse mapeamento (`LABEL_MAP`) é o próximo passo, antes de
  comparar diretamente com os resultados do P2.
- Para ablação (com/sem contexto, `mask_digits=True/False`), troque os parâmetros de
  `BertEmbedder` e rode de novo: o cache é separado por configuração (`embedder.name`).